<a href="https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit

**Lane:** CTR / Engagement Opportunity Scoring

This notebook audits candidate February signals before modeling. The goal is not to claim causality; it is to establish whether the measured signals are useful and directionally connected to the March CTR outcome.

## 1. Load the February features and March outcome

The audit follows the ML-04 data contract:

- Features: February 2026
- Outcome: March 2026
- Eligible content: at least 100 March impressions
- Future CTR = March clicks / March impressions
- Position 0 is treated as unavailable
- Client IDs are grouping variables, not predictive features

In [5]:
import os
import duckdb
import pandas as pd
import numpy as np

os.makedirs("work/outputs", exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"

# Use a DuckDB session variable so the token is not interpolated into SQL text.
if HF_TOKEN:
    con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
    con.execute(
        "CREATE OR REPLACE SECRET hf "
        "(TYPE huggingface, TOKEN getvariable('hf_token'))"
    )

print("Warehouse paths configured: February features + March outcome")

audit_sql = f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feb_impressions,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feb_clicks,

        100.0 * SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        )
        / NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS feb_ctr,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                     AND gsc_avg_position > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        / NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                         AND gsc_avg_position > 0
                THEN gsc_impressions
                ELSE 0
            END
        ), 0) AS feb_avg_position

    FROM read_parquet('{FEB}')

    WHERE report_date BETWEEN DATE '2026-02-01'
                          AND DATE '2026-02-28'

    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS march_impressions,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS march_clicks,

        100.0 * SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        )
        / NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS future_ctr

    FROM read_parquet('{MAR}')

    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-31'

    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.*,
    mar.march_impressions,
    mar.march_clicks,
    mar.future_ctr

FROM feb

JOIN mar
    USING (client_hash_id, content_hash_id)

WHERE mar.march_impressions >= 100
"""

audit = con.sql(audit_sql).df()

SIGNALS = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position"
]

print("Eligible rows:", len(audit))
print("Signals under audit:", SIGNALS)
print()
print(audit[SIGNALS + ["future_ctr"]].head())

Warehouse paths configured: February features + March outcome


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 93671
Signals under audit: ['feb_impressions', 'feb_clicks', 'feb_ctr', 'feb_avg_position']

   feb_impressions  feb_clicks  feb_ctr  feb_avg_position  future_ctr
0            235.0         0.0      0.0          6.391489         0.0
1            660.0         0.0      0.0         61.680303         0.0
2           1139.0         0.0      0.0         49.698859         0.0
3           1063.0         0.0      0.0         54.554092         0.0
4            586.0         0.0      0.0         42.027304         0.0


## 2. Basic signal quality

We first inspect scale, missingness, and whether the signals contain usable variation. A useful signal must be available at prediction time and have enough observed variation to support ranking.

In [6]:
quality = pd.DataFrame({
    "missing": audit[SIGNALS].isna().sum(),
    "missing_pct": 100 * audit[SIGNALS].isna().mean(),
    "n_unique": audit[SIGNALS].nunique(),
    "median": audit[SIGNALS].median(),
})
print(quality.round(3))

print("\nFuture CTR summary:")
print(audit["future_ctr"].describe().round(3))

                  missing  missing_pct  n_unique   median
feb_impressions         0        0.000     10920  478.000
feb_clicks              0        0.000       347    1.000
feb_ctr              7111        7.591     21929    0.106
feb_avg_position     7186        7.672     78421    7.366

Future CTR summary:
count    93671.000
mean         0.248
std          0.400
min          0.000
25%          0.000
50%          0.118
75%          0.350
max         15.584
Name: future_ctr, dtype: float64


## 3. Directional signal audit

For this lane, two signals are especially relevant:

- **Position:** CTR generally differs across search-position tiers, so position can provide context for whether a low CTR is unusual for the page's visibility level.
- **Impression volume:** higher measured volume gives more evidence about the observed CTR and makes a potential opportunity more actionable.

The audit uses rank/correlation-style evidence only. It does **not** interpret these relationships as causal.

In [7]:
audit["position_bin"] = pd.cut(
    audit["feb_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["Top 3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

position_summary = (
    audit.groupby("position_bin", observed=False)
    .agg(
        rows=("future_ctr", "size"),
        feb_ctr=("feb_ctr", "mean"),
        future_ctr=("future_ctr", "mean"),
        impressions=("feb_impressions", "median")
    )
    .reset_index()
)

print(position_summary.round(3).to_string(index=False))

print("\nSpearman-style rank correlations with future CTR:")
for col in SIGNALS:
    pair = audit[[col, "future_ctr"]].dropna()
    print(f"{col:18s} n={len(pair):6d}  rho={pair[col].corr(pair['future_ctr'], method='spearman'):.4f}")

position_bin  rows  feb_ctr  future_ctr  impressions
       Top 3 11194    0.354       0.292       1207.0
        4-10 44486    0.334       0.280        687.0
       11-20 17237    0.257       0.208        384.0
       21-50 11523    0.161       0.117        301.0
         50+  2045    0.176       0.050         81.0

Spearman-style rank correlations with future CTR:
feb_impressions    n= 93671  rho=0.2707
feb_clicks         n= 93671  rho=0.4569
feb_ctr            n= 86560  rho=0.5031
feb_avg_position   n= 86485  rho=-0.2732


## 4. Volume and CTR signal checks

CTR is a ratio, so impression volume matters when interpreting observed CTR. We therefore inspect the relationship between February impressions and the later outcome, while keeping the interpretation directional.

In [8]:
audit["impression_bin"] = pd.qcut(
    audit["feb_impressions"].rank(method="first"),
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
)

volume_summary = (
    audit.groupby("impression_bin", observed=False)
    .agg(
        rows=("future_ctr", "size"),
        median_feb_impressions=("feb_impressions", "median"),
        mean_feb_ctr=("feb_ctr", "mean"),
        mean_future_ctr=("future_ctr", "mean")
    )
    .reset_index()
)

print(volume_summary.round(3).to_string(index=False))

impression_bin  rows  median_feb_impressions  mean_feb_ctr  mean_future_ctr
            Q1 18735                    13.0         0.379            0.267
            Q2 18734                   172.0         0.258            0.207
            Q3 18734                   478.0         0.247            0.216
            Q4 18734                  1292.0         0.288            0.254
            Q5 18734                  4531.0         0.330            0.296


## 5. Audit conclusion

**Keep:** `feb_ctr`, `feb_avg_position`, `feb_impressions`, and `feb_clicks` as candidate signals.

**Why:** they are available before the March outcome and represent distinct information about measured search performance and evidence volume.

**Important:** this is a signal audit, not proof that changing any feature will cause CTR to improve. Position and CTR are observationally associated and may be affected by other factors.

The later baseline and model stages decide whether these signals improve ranking quality on held-out clients.

In [9]:
# Machine-checkable summary for the next stage.
assert all(c in audit.columns for c in SIGNALS + ["future_ctr"])
assert "trend_direction" not in SIGNALS
assert "trend_pct" not in SIGNALS
assert "future_ctr" not in SIGNALS

print("PASS: all audited signals are pre-outcome features; future_ctr is outcome-only.")
print("Candidate signals:", ", ".join(SIGNALS))

PASS: all audited signals are pre-outcome features; future_ctr is outcome-only.
Candidate signals: feb_impressions, feb_clicks, feb_ctr, feb_avg_position


## Self-check

- [x] Candidate signals are explicitly listed.
- [x] Availability and missingness are checked.
- [x] Directional relationships are audited.
- [x] No causal claim is made.
- [x] Future outcome is kept separate from features.
- [ ] Run **Runtime → Run all** in Colab and save the executed notebook back to the exact repo path.